<a href="https://colab.research.google.com/github/NicolasPetiot/EnjeuxDecarbonationSante/blob/main/rosetta_MCSS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
from pathlib import Path

if "google.colab" in sys.modules:

    from google.colab import drive
    drive.mount('/content/drive')

    DATA_DIR = Path("/content/drive/MyDrive/EnjeuxDecarbonationSante/")

else:
    DATA_DIR = Path(".")


if not DATA_DIR.exists():
  raise FileNotFoundError(f"Le répertoire '{DATA_DIR}' n'est pas accessible ou n'existe pas")

In [ ]:
def rosetta_setup(verbosity = False, allow_overwrite = True, H_optimization = True, extra_params:list[Path] = []):
    flags = [
        f"-mute false" if verbosity else "-mute all",
        f"-ex1",
        f"-ex2",
        f"-no_optH {str(not H_optimization)}",
        f"-flip_HNQ true",
        f"-ignore_ligand_chi true",
        f"-overwrite" if allow_overwrite else "",
        f"-restore_pre_talaris_2013_behavior true"
    ]
    flags = " ".join(flags)

    extra_flag = []
    for path in extra_params:
        if not path.exists():
            print(f"WARNING: ignoring file {str(path)} (does not exists or is not acessible)")

        else:
            extra_flag.append(str(path))
    if len(extra_flag) > 0:
        flags += " -extra_res_fa " + " ".join(extra_flag)

    try:
        from pyrosetta import init

    except ImportError:
        !pip install pyrosetta_installer
        from pyrosetta_installer import install_pyrosetta
        install_pyrosetta()

        from pyrosetta import init

    finally:
        init(flags)

def load_rosetta_pose(path:Path):
    from pyrosetta import pose_from_file
    if path.exists():
        return pose_from_file(str(path))

    else:
        raise FileNotFoundError(f"Le fichier {str(path)} n'existe pas ou n'est pas accessible.")

In [ ]:
rosetta_setup(extra_params=[
    DATA_DIR / ".rosettafiles/GSH.params"
], verbosity=False, H_optimization=True)

In [ ]:
## Define Mutation Protocol:
from pyrosetta import Pose
from pyrosetta.rosetta.protocols.rosetta_scripts import XmlObjects

MUTATE_XML_STRING = """
<ROSETTASCRIPTS>
	<SCOREFXNS>
	</SCOREFXNS>
	<RESIDUE_SELECTORS>
        <Index name="to_mutate_A" resnums="{resi}A"/>
        <Index name="to_mutate_B" resnums="{resi}B"/>
        <Or name="to_mutate" selectors="to_mutate_A,to_mutate_B" />
	</RESIDUE_SELECTORS>
	<MOVERS>
        <MutateResidue name="mutate" residue_selector="to_mutate" new_res="{new_res}" preserve_atom_coords="false" />
	</MOVERS>
	<PROTOCOLS>
        <Add mover_name="mutate" />
	</PROTOCOLS>
	<OUTPUT />
</ROSETTASCRIPTS>
"""

def mutate(pose:Pose, aa_num:int, new_aa:str) -> Pose:
    """
    Apply a mutation to both chains A&B of an input pose.

    The mutation site is identified using an input residue index `resi:int`

    The new residue is specified using the single letter residue code `new_res`
    """
    xml = XmlObjects.create_from_string(MUTATE_XML_STRING.format(resi=aa_num, new_res=new_aa))
    protocol = xml.get_mover("ParsedProtocol")

    mutant = pose.clone()
    protocol.apply(mutant)
    return mutant

In [ ]:
## Define Binding Affinity Protocol:
from pyrosetta import Pose, ScoreFunction, Vector1
from pyrosetta import get_fa_scorefxn
from pyrosetta.rosetta import protocols

def binding_affinity(pose:Pose, partners = "AB_C", scorefxn:ScoreFunction = None) -> float:
    """
    Separates two partners and returns the difference of energy between bounded and separated states.
    """
    if scorefxn is None:
        scorefxn = get_fa_scorefxn()

    bind_score = scorefxn(pose)

    # Split partners:
    split_pose = pose.clone()
    jump = 2
    step_size = 100

    protocols.docking.setup_foldtree(pose, partners, Vector1([-1,-1,-1]))
    trans_mover = protocols.rigid.RigidBodyTransMover(split_pose,jump)
    trans_mover.step_size(step_size)
    trans_mover.apply(split_pose)

    split_score = scorefxn(split_pose)

    return bind_score - split_score

In [ ]:
## Define Random mutation selection:
import random

MUTATION_SITES = [7, 12, 13, 34, 39, 40, 48, 50, 51, 52, 53, 54, 55, 64, 65, 66, 67, 69, 102, 106, 114]
AMINO_ACIDS = ["ARG", "HIS", "LYS", "ASP", "GLU", "SER", "THR", "ASN", "GLN", "CYS", "GLY", "PRO", "ALA", "VAL", "ILE", "LEU", "MET", "PHE", "TYR", "TRP"]

def select_random_mutation(pose:Pose, force_change = False):
    three_letter_code ={'V':'VAL', 'I':'ILE', 'L':'LEU', 'E':'GLU', 'Q':'GLN',
    'D':'ASP', 'N':'ASN', 'H':'HIS', 'W':'TRP', 'F':'PHE', 'Y':'TYR',
    'R':'ARG', 'K':'LYS', 'S':'SER', 'T':'THR', 'M':'MET', 'A':'ALA',
    'G':'GLY', 'P':'PRO', 'C':'CYS'}

    # Select mutation site:
    resi = random.choice(MUTATION_SITES)

    old_aa = pose.sequence()[resi-1] # One-letter code
    old_aa = three_letter_code[old_aa]

    allowed_residues = AMINO_ACIDS.copy()
    if force_change:
        allowed_residues.remove(old_aa)

    new_aa = random.choice(allowed_residues)

    return old_aa, resi, new_aa

In [ ]:
pdb = DATA_DIR / "PDB/GSTD1+GSH.pdb"
ref = load_rosetta_pose(pdb)
dG_ref = binding_affinity(ref)

old, resi, new = select_random_mutation(ref, force_change=True)

mutant = mutate(ref, resi, new)
dG_mutant = binding_affinity(mutant)

ddG = dG_mutant - dG_ref
print(f"Mutation ddG: {ddG:.3f} R.E.U.")

In [ ]:
## Define Metropolis Criterion:
from math import exp
import random

def metropolis(delta:float, temp:float) -> bool:
    """

    """
    if delta < 0:
        return True

    if temp == 0.0:
        return False

    acceptation_probability = exp(-delta/temp)
    return random.uniform(0, 1) < acceptation_probability

In [ ]:
from tqdm.notebook import tqdm

import pandas as pd

N_ITER = 500
PDB_INIT = DATA_DIR / "PDB/GSTD1+GSH.pdb"
TEMP = 0.2

# Initialisation:
ref = load_rosetta_pose(PDB_INIT)
dG_ref = binding_affinity(ref)

old_aa = [pd.NA]
indices = [pd.NA]
new_aa = [pd.NA]

dGs = [dG_ref]
ddGs = [pd.NA]
accepted = [True]

min_dG = float("inf") # Utile pour sauvegarder le meilleur design
save_mutant = False

for _ in tqdm(range(N_ITER)):
    # 1- Selection aléatoire de la mutation:
    old, resi, new = select_random_mutation(ref, force_change=True)
    mutant = mutate(ref, resi, new)

    # 2- Affinité et ddG:
    dG_mutant = binding_affinity(mutant)
    ddG = dG_mutant - dG_ref

    # 3- Critère de Métropolis:
    is_accepted = metropolis(ddG, temp=TEMP)
    if is_accepted:
        ref = mutant.clone()
        dG_ref = dG_mutant

    # 4- Sauvegarde des informations pertinentes:
    old_aa.append(old)
    indices.append(resi)
    new_aa.append(new)

    dGs.append(dG_mutant)
    ddGs.append(ddG)
    accepted.append(is_accepted)

    if dG_mutant < min_dG:
        min_dG = dG_mutant
        best_pose = mutant.clone()

df = pd.DataFrame({
    "old": old_aa,
    "resi": indices,
    "new": new_aa,

    "dG": dGs,
    "ddG": ddGs,
    "accepted": accepted
})

if save_mutant:
    mutant_filename = str(PDB_INIT).replace(".pdb", f"_best_design.pdb")

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.figsize": (8, 3),
    "font.size": 14
})

fig, ax = plt.subplots()

sele = df.query("accepted")
ax.scatter(sele.index, sele.dG, s=20, c="forestgreen", ec="k", lw=0.25, zorder = 2)

ylim = ax.get_ylim()

sele = df.query("not accepted")
ax.scatter(sele.index, sele.dG, s=10, c="firebrick", ec="k", lw=0.25)

ax.set_ylim(ylim)
ax.set_xlabel("Itération MCSS")
ax.set_ylabel("$\\Delta G$ [R.E.U.]")